# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
import os, sys, math, time, gc

# ── Global Path Configuration ──────────────────
# We look for checkpoints in Inputs (for Resume) but always save to Working
SAVE_PATH = "/kaggle/working/ppc_3b_pretrain.pt"
LOAD_PATH = SAVE_PATH # Default

# ── Priority Load Logic (Input > Working) ──────────────────
if os.path.exists("/kaggle/input"):
    import glob
    possible_files = glob.glob("/kaggle/input/**/ppc_3b_pretrain.pt", recursive=True)
    if possible_files:
        LOAD_PATH = possible_files[0]
        print(f"🚀 Priority Found (Version Output): {LOAD_PATH}")
    elif os.path.exists(SAVE_PATH):
        LOAD_PATH = SAVE_PATH
        print(f"📦 Using Local/Rescued Checkpoint: {LOAD_PATH}")
else:
    LOAD_PATH = SAVE_PATH

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Essential Dependencies (These are usually missing on Kaggle)
!pip install -q bitsandbytes wandb datasets transformers accelerate tiktoken

# 2. Pull Research Source (Injecting into path instead of installing)
REPO_NAME = 'EFV-nn'
REPO_URL = 'https://github.com/ey3lock3r/EFV-nn.git'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone -q {REPO_URL}')
else:
    os.system(f'cd {REPO_NAME} && git pull -q')

    sys.path.insert(0, f'{REPO_NAME}/src') # Force local source priority

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import bitsandbytes as bnb
import wandb
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from kaggle_secrets import UserSecretsClient

from efv_nn.ppc_sharded import ShardedPPCGraphLLM

print("✅ Environment ready. Source loaded from GitHub.")
print(f"CUDA Available: {torch.cuda.is_available()} | Devices: {torch.cuda.device_count()}")


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print("✅ Logged into W&B and HF")
except Exception as e:
    print(f"⚠️ Secrets failure: {e}. Training will continue without W&B.")


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ───────────────
def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex["text"])["input_ids"])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


In [ ]:
# ── Cell 4: Optional 🛟 W&B Rescue (Pull checkpoint from cloud) ──────────
import shutil, os, wandb
RESCUE_RUN_PATH = "dhinsresearch/ppc-3b-dynamic/YOUR_RUN_ID" # Update this
RESCUE_FILE = "ppc_3b_pretrain.pt"

def rescue_checkpoint(run_path, filename, target_path):
    print(f"📡 Synching {filename} from {run_path}...")
    try:
        restored_file = wandb.restore(filename, run_path=run_path)
        source_path = os.path.abspath(restored_file.name)
        target_path = os.path.abspath(target_path)
        if source_path != target_path:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            shutil.move(source_path, target_path)
            print(f"🚚 Moved from {source_path} → {target_path}")
        print(f"✅ Rescue Successful! Ready at: {target_path}")
    except Exception as e:
        print(f"❌ Rescue Failed: {e}")

# rescue_checkpoint(RESCUE_RUN_PATH, RESCUE_FILE, SAVE_PATH)


In [ ]:
# ── Cell 5: 🏗️ Architectural Setup (Run ONCE) ────────────
import importlib
import efv_nn.triton_kernels, efv_nn.ppc_sharded, efv_nn.ppc_gnn, efv_nn.ppc_core
importlib.reload(efv_nn.triton_kernels) # Force Hyper-Drive reload
print(f'🔍 Loading Kernels from: {efv_nn.triton_kernels.__file__}')
importlib.reload(efv_nn.ppc_core)
importlib.reload(efv_nn.ppc_gnn)
importlib.reload(efv_nn.ppc_sharded)
from efv_nn.ppc_sharded import ShardedPPCGraphLLM

VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
PRIME_DELAYS = [1, 2, 3, 5, 7, 11, 13, 17] # Maximum Phasal IQ (8 delays)
RESUME = True


print('📥 Loading Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', token=os.environ.get('HF_TOKEN'))

print('🏗️ Instantiating 3.2B Sharded Model...')
gc.collect(); torch.cuda.empty_cache()
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, use_jacobian=True, prime_delays=PRIME_DELAYS)

if RESUME and os.path.exists(LOAD_PATH):
    print(f'🔄 Loading Checkpoint: {LOAD_PATH}')
    ckpt = torch.load(LOAD_PATH, map_location='cpu')
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    model.load_state_dict(state_dict, strict=False)
    START_STEP = ckpt.get('step', 0)
    print(f'✅ Resumed Weight Step {START_STEP}')

# Guard: store device refs
DEVICE1 = model.device1
if 'START_STEP' not in locals(): START_STEP = 0

raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
triton_ok = getattr(raw_model.layers[0], '_triton_available', False)
t_msg = 'ACTIVE (Hyper-Drive)' if triton_ok else 'FALLBACK'
print(f'⚡ Triton: {t_msg}')
print(f'✅ Model Ready. Total Params: {model.total_params/1e9:.2f}B')


In [ ]:
# ── Cell 6: 🔬 Model Identity Audit (Sanity Check) ──
def run_identity_audit(model, path):
    def _audit_weights(label, weights):
        w_mean = weights.abs().mean().item()
        w_std  = weights.std().item()
        status = "✅ LEARNED" if w_std > 0.05 else "❌ RANDOM/NOISE"
        print(f"[{label}] Mean: {w_mean:.6f} | Std: {w_std:.6f} | Status: {status}")
        return w_std > 0.05

    print("🔍 Auditing Memory vs Disk...")
    # 1. Memory
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    memory_learned = _audit_weights("MEMORY", raw_model.embedding.weight)
    
    # 2. Disk
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location='cpu')
            s_dict = ckpt['model_state'] if 'model_state' in ckpt else ckpt
            w_disk = s_dict.get('embedding.weight') or s_dict.get('_orig_mod.embedding.weight')
            if w_disk is not None:
                _audit_weights("DISK  ", w_disk.float())
            else:
                print("⚠️ DISK: Could not find 'embedding.weight' in checkpoint.")
        except Exception as e:
            print(f"⚠️ DISK: Failed to read checkpoint: {e}")
    else:
        print("⚪ DISK: No checkpoint found at target path.")
    
    if not memory_learned:
        print("\n❗ WARNING: Model in memory is RANDOM. Please verify your load logic.")

# 🚀 To perform a sanity check, uncomment the line below:
# run_identity_audit(model, LOAD_PATH)


In [ ]:
# ── Cell 7: 🚀 Start Training Loop (Stabilized) ──
ENABLE_SPECTRAL_GUARDIAN = True  # Toggle here for easier experimentation

from efv_nn.ppc_gnn import spectral_guardian_penalty
import shutil

def save_checkpoint(model, step, path, avg_energy=0.0, energy_threshold=1.0, sync_wandb=True):
    """
    Saves weights with Corruption Blocking and Atomic Rotation.
    """
    if avg_energy > energy_threshold:
        print(f"🚫 ABORTED: Energy {avg_energy:.4f} exceeds threshold ({energy_threshold}). Protecting healthy checkpoint.")
        return

    gc.collect(); torch.cuda.empty_cache()
    
    # 1. Atomic Rotation: Keep a 'prev' version in case of silent corruption
    if os.path.exists(path):
        prev_path = path.replace(".pt", "_prev.pt")
        shutil.copy(path, prev_path)
        # Also inform wandb about the prev version (base_path ensures it lands in root)
        if sync_wandb:
            wandb.save(prev_path, base_path=os.path.dirname(prev_path))

    # 2. Extract and Compress
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    state_dict = raw_model.state_dict()
    # Force float() before half() to ensure phasal balance in interleaved pairs
    compressed_dict = {k: (v.float().half() if v.dtype == torch.float32 else v) for k, v in state_dict.items()}
    
    # 3. Save Locally
    torch.save({'step': step, 'model_state': compressed_dict}, path)
    
    # 4. Sync to Cloud Root
    if sync_wandb:
        wandb.save(path, base_path=os.path.dirname(path))
    print(f"✅ Step {step} saved. (Energy: {avg_energy:.4f})")

def train(model, tokenizer, num_steps=1000, start_step=0, local_iters=24, lr=8e-5):
    gc.collect(); torch.cuda.empty_cache()

    wandb.init(project="ppc-3b-dynamic", name=f"ppc-3b-{time.strftime('%H%M')}")
    dataloader = get_dataloader(tokenizer, batch_size=2, seq_len=256)
    delay_params = []
    base_params = []
    for name, p in model.named_parameters():
        if "delay_gains" in name:
            delay_params.append(p)
        else:
            base_params.append(p)
            
    opt = bnb.optim.PagedAdamW8bit([
        {'params': base_params, 'lr': lr},
        {'params': delay_params, 'lr': 5e-4}  # Hot start for OCNS Causal Memory
    ])

    t0 = time.time(); last_step = start_step

    guardian_status = '🛡️ ON' if ENABLE_SPECTRAL_GUARDIAN else '⚪ OFF'
    print(f"🚀 Training [Iters: {local_iters}, LR: {lr}, Spectral Guardian: {guardian_status}]...")

    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        ids, targets = batch[:, :-1], batch[:, 1:].to(DEVICE1)

        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, avg_iters, avg_energy, layer_energies = model(ids, local_iters=local_iters)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

        # Spectral Guardian: OUTSIDE autocast to preserve FP32 precision
        if ENABLE_SPECTRAL_GUARDIAN:
            loss = loss + spectral_guardian_penalty(layer_energies.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % 5 == 0:
            ppl = math.exp(min(loss.item(), 20))
            lr_curr = opt.param_groups[0]['lr']
            dt = (time.time() - t0) * 1000 / (step - last_step) if step > last_step else 0
            t0 = time.time(); last_step = step
            wandb.log({"loss": loss.item(), "ppl": ppl, "avg_iters": avg_iters,
                       "avg_energy": avg_energy.item(), "duration_ms": dt}, step=step)
            print(f"St {step} | L: {loss.item():.4f} | E: {avg_energy.item():.4f} | D: {dt:.0f}ms")

        if (step + 1) % 100 == 0:
            save_checkpoint(model, step + 1, SAVE_PATH, avg_energy=avg_energy.item(), sync_wandb=((step + 1) % 500 == 0))
            t0 = time.time(); last_step = step  # Reset timer: exclude save I/O from D_avg
            
        if step >= (start_step + num_steps): break

    print(f"\n[PPC-GTB Snapshot] Finished at Step {step}")
    print(f"L: {loss.item():.4f} | E: {avg_energy.item():.4f} | D_avg: {dt:.0f}ms")
    wandb.finish()


In [ ]:
# ── Cell 8: 🚀 Start Training Loop (Axiomatic Phases) ──

# Phase 1: 🛡️ Stabilization (16 iters, 1.5e-4, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=500, local_iters=16, lr=1.5e-4)

# Phase 2: 🚀 Depth Scaling (24 iters, 1.5e-4, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=2000, local_iters=24, lr=1.5e-4)

# Phase 3: 💎 Precision Polish (32 iters, 5e-5, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=5000, local_iters=32, lr=5e-5)

# Phase 4: 🧠 Semantic Emergence (32 iters, 5e-5, RELEASED GRADIENTS)
# Completed at Step 10,000. E climbed to 0.0034 — plateau confirmed.
# train(model, tokenizer, start_step=START_STEP, num_steps=10000, local_iters=32, lr=5e-5)

# Phase 5: 🔬 Deep Resonance Tuning (48 iters, 1e-5, RELEASED GRADIENTS)
# Lower LR cures energy creep (0.0034 → 0.0024). Higher iters break loss plateau.
# Check-in at Step 12,000 (2,000 steps): verify E < 0.003 and loss resuming downtrend.
train(model, tokenizer, start_step=START_STEP, num_steps=10000, local_iters=48, lr=1e-5)


In [ ]:
# ── Cell 9: 🗣️ Verification (Instant) ──
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    out_reg = model.generate(inputs, max_new_tokens=40, local_iters=32)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    out_swarm = model.generate_swarm(inputs, max_new_tokens=40, swarm_size=4, local_iters=32)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")
    print("\n🧬 Running MoE Health Audit...")
    captured_indices = []
    def hook_fn(module, input, output):
        captured_indices.append(output[1].detach().cpu())
        
    hooks = []
    try:
        for name, module in model.named_modules():
            if module.__class__.__name__ == 'ExpertChoiceMoEMatcher':
                hooks.append(module.register_forward_hook(hook_fn))
                
        with torch.no_grad(), torch.amp.autocast('cuda'):
            model(inputs, local_iters=8)
            

    finally:
        for h in hooks: h.remove()
    
    B_T = inputs.shape[1]
    total_coverage = 0
    for indices in captured_indices[:24]:
        flat_indices = indices.flatten()
        unique_tokens = torch.unique(flat_indices).shape[0]
        total_coverage += (unique_tokens / float(B_T))
            
    avg_cov = total_coverage / len(captured_indices[:24])
    print(f"🔬 Average Expert Diversity Score (Coverage): {avg_cov:.1%}")
    if avg_cov < 0.3:
        print("🚨 WARNING: High Expert Cloning detected. Experts are picking the same tokens.")
    elif avg_cov > 0.8:
        print("✅ HEALTHY: Maximum MoE utilization. Experts are highly specialized.")
    else:
        print("⚠️ MODERATE: Experts are learning, but there is noticeable overlap.")


# To run: 
# verify_generation(model, tokenizer)


In [ ]:
# ── Cell 10: 🗣️ Generation Verification ──
def verify_generation(prompt="The fundamental principles of biology include"):
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))
    model = ShardedPPCGraphLLM(128256, 1024, 24, 64, prime_delays=PRIME_DELAYS)
    # Check LOAD_PATH first for verification
    target_ckpt = LOAD_PATH if os.path.exists(LOAD_PATH) else SAVE_PATH
    if os.path.exists(target_ckpt):
        print(f"🔍 Verifying checkpoint from: {target_ckpt}")
        ckpt = torch.load(target_ckpt, map_location='cpu')
        ckpt = torch.load(LOAD_PATH, map_location='cpu')
        state_dict = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    model.load_state_dict(state_dict, strict=False)
    START_STEP = ckpt.get('step', 0)
    print(f"✅ Resumed Weight Step {START_STEP}")
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    print("\n🧠 Running Regular Generation...")
    output_ids = model.generate(inputs, max_new_tokens=50, local_iters=32)
    print(f"Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")
    
    print("\n🐝 Running Advanced Swarm Inference (N=8)...")
    swarm_ids = model.generate_swarm(inputs, max_new_tokens=50, swarm_size=4, local_iters=32)
    print(f"Output:\n{tokenizer.decode(swarm_ids[0], skip_special_tokens=True)}")
    print(f"\nPrompt: {prompt}")
    print("\n🧬 Running MoE Health Audit...")
    captured_indices = []
    def hook_fn(module, input, output):
        captured_indices.append(output[1].detach().cpu())
        
    hooks = []
    try:
        for name, module in model.named_modules():
            if module.__class__.__name__ == 'ExpertChoiceMoEMatcher':
                hooks.append(module.register_forward_hook(hook_fn))
                
        with torch.no_grad(), torch.amp.autocast('cuda'):
            model(inputs, local_iters=8)
            

    finally:
        for h in hooks: h.remove()
    
    B_T = inputs.shape[1]
    total_coverage = 0
    for indices in captured_indices[:24]:
        flat_indices = indices.flatten()
        unique_tokens = torch.unique(flat_indices).shape[0]
        total_coverage += (unique_tokens / B_T)
            
    avg_cov = total_coverage / len(captured_indices[:24])
    print(f"🔬 Average Expert Diversity Score (Coverage): {avg_cov:.1%}")
    if avg_cov < 0.3:
        print("🚨 WARNING: High Expert Cloning detected. Experts are picking the same tokens.")
    elif avg_cov > 0.8:
        print("✅ HEALTHY: Maximum MoE utilization. Experts are highly specialized.")
    else:
        print("⚠️ MODERATE: Experts are learning, but there is noticeable overlap.")


# verify_generation()

